In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
%%writefile matmul_tiling.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cstdlib>
#include <iostream>

#define TILE 32

__global__ void matmul_naive(float* A,float* B,float* C,int N){
    __shared__ float As[TILE*TILE];
    __shared__ float Bs[TILE*TILE];

    int globalcol=blockIdx.x*blockDim.x+threadIdx.x;
    int globalrow=blockIdx.y*blockDim.y+threadIdx.y;
    int localrow=threadIdx.y;
    int localcol=threadIdx.x;
    int NUM_TILES=(N+TILE-1)/TILE;
    float sum=0.0f;
    for(int i=0;i<NUM_TILES;i++){
        int Arow=globalrow;
        int Acol=i*TILE+localcol;
        int Brow=i*TILE+localrow;
        int Bcol=globalcol;

        if(Arow<N && Acol<N){
            As[localrow*TILE+localcol]=A[Arow*N+Acol]; 
        }
        else{
            As[localrow*TILE+localcol]=0.0f;
        }
        if(Brow<N && Bcol<N){
            Bs[localrow*TILE+localcol]=B[Brow*N+Bcol];
        }
        else{
            Bs[localrow*TILE+localcol]=0.0f;
        }
        __syncthreads();

        for(int i=0;i<TILE;i++){
            sum+=As[localrow*TILE+i]*Bs[i*TILE+localcol];
        }        
        __syncthreads();
    }
    if(globalrow<N && globalcol<N){
        C[globalrow*N+globalcol]=sum;
    }
}

using namespace std; 

int main(int argc,char* argv[]){
    if (argc < 2) {
        printf("Usage: ./matmul_naive N\n");
        return 1;
    }

    int N = atoi(argv[1]);
    size_t size = N * N * sizeof(float);

    printf("Running GPU naive matmul for N = %d\n", N);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N*N; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 1.0f;
    }
    cudaEvent_t start, stop, start_overall, stop_overall;
    cudaEventCreate(&start_overall);
    cudaEventCreate(&stop_overall);
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    cudaEventRecord(start_overall);
    
    float *d_A,*d_B,*d_C;
    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);
    
    cudaMemcpy(d_A,h_A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,size,cudaMemcpyHostToDevice);

    dim3 ThreadsPerBlock(32,32);
    int t=(N+32-1)/32;
    dim3 BlocksPerGrid(t,t);

    cudaDeviceSynchronize();

    cudaEventRecord(start);
    matmul_naive<<<BlocksPerGrid,ThreadsPerBlock>>>(d_A,d_B,d_C,N);
    cudaEventRecord(stop);

    cudaDeviceSynchronize();
    cudaMemcpy(h_C,d_C,size,cudaMemcpyDeviceToHost);
    
    cudaEventRecord(stop_overall);

    cudaEventSynchronize(stop);
    float kernelmilliseconds = 0;
    cudaEventElapsedTime(&kernelmilliseconds, start, stop);
    printf("Kernel Execution time: %.3f ms\n", kernelmilliseconds);

    cudaEventSynchronize(stop_overall);
    float overallmilliseconds = 0;
    cudaEventElapsedTime(&overallmilliseconds, start_overall, stop_overall);
    printf("Total execution time on GPU: %.3f ms\n", overallmilliseconds);
    
    cudaEventDestroy(start_overall);
    cudaEventDestroy(stop_overall);
    
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            if(h_C[i*N+j]-N > 1e-5){
                fprintf(stderr, "Result verification failed at element %d!\n", i);
                cudaFree(d_A);
                cudaFree(d_B);
                cudaFree(d_C);
                free(h_A);
                free(h_B);
                free(h_C);
                exit(EXIT_FAILURE);
            }
        }
    }
    cout<<"Works Correctly"<<endl;
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);
    return 0;
}

Overwriting matmul_tiling.cu


In [8]:
!nvcc matmul_tiling.cu -o ./matmul_tiling

In [3]:
!nvprof ./matmul_tiling 1024

Running GPU naive matmul for N = 1024
==182== NVPROF is profiling process 182, command: ./matmul_tiling 1024
Kernel Execution time: 137.136 ms
Total execution time on GPU: 144.250 ms
Works Correctly
==182== Profiling application: ./matmul_tiling 1024
==182== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   49.09%  4.0606ms         1  4.0606ms  4.0606ms  4.0606ms  matmul_naive(float*, float*, float*, int)
                   32.11%  2.6564ms         1  2.6564ms  2.6564ms  2.6564ms  [CUDA memcpy DtoH]
                   18.80%  1.5548ms         2  777.42us  754.36us  800.48us  [CUDA memcpy HtoD]
      API calls:   58.19%  211.19ms         4  52.798ms     517ns  211.19ms  cudaEventCreate
                   36.67%  133.07ms         1  133.07ms  133.07ms  133.07ms  cudaLaunchKernel
                    1.84%  6.6666ms         3  2.2222ms  985.64us  4.2820ms  cudaMemcpy
                    1.50%  5.4347ms       228  23.836us

In [4]:
!nvprof ./matmul_tiling 2048

Running GPU naive matmul for N = 2048
==203== NVPROF is profiling process 203, command: ./matmul_tiling 2048
Kernel Execution time: 40.240 ms
Total execution time on GPU: 61.702 ms
Works Correctly
==203== Profiling application: ./matmul_tiling 2048
==203== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   67.83%  39.963ms         1  39.963ms  39.963ms  39.963ms  matmul_naive(float*, float*, float*, int)
                   20.56%  12.116ms         1  12.116ms  12.116ms  12.116ms  [CUDA memcpy DtoH]
                   11.61%  6.8377ms         2  3.4189ms  3.4056ms  3.4321ms  [CUDA memcpy HtoD]
      API calls:   73.02%  184.71ms         4  46.178ms     544ns  184.71ms  cudaEventCreate
                   15.83%  40.055ms         2  20.027ms  85.200us  39.969ms  cudaDeviceSynchronize
                    8.30%  20.994ms         3  6.9980ms  3.5631ms  13.752ms  cudaMemcpy
                    1.92%  4.8650ms       228  21.33

In [5]:
!nvprof ./matmul_tiling 4096

Running GPU naive matmul for N = 4096
==224== NVPROF is profiling process 224, command: ./matmul_tiling 4096
Kernel Execution time: 217.523 ms
Total execution time on GPU: 303.149 ms
Works Correctly
==224== Profiling application: ./matmul_tiling 4096
==224== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   72.46%  217.27ms         1  217.27ms  217.27ms  217.27ms  matmul_naive(float*, float*, float*, int)
                   17.94%  53.800ms         1  53.800ms  53.800ms  53.800ms  [CUDA memcpy DtoH]
                    9.59%  28.762ms         2  14.381ms  14.379ms  14.383ms  [CUDA memcpy HtoD]
      API calls:   43.37%  217.36ms         2  108.68ms  83.675us  217.28ms  cudaDeviceSynchronize
                   38.01%  190.48ms         4  47.620ms     502ns  190.47ms  cudaEventCreate
                   16.98%  85.091ms         3  28.364ms  14.553ms  55.926ms  cudaMemcpy
                    1.00%  4.9993ms       228  21.

In [6]:
!nvprof ./matmul_tiling 8192

Running GPU naive matmul for N = 8192
==245== NVPROF is profiling process 245, command: ./matmul_tiling 8192
Kernel Execution time: 1227.518 ms
Total execution time on GPU: 1563.383 ms
Works Correctly
==245== Profiling application: ./matmul_tiling 8192
==245== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   78.64%  1.22726s         1  1.22726s  1.22726s  1.22726s  matmul_naive(float*, float*, float*, int)
                   14.07%  219.60ms         1  219.60ms  219.60ms  219.60ms  [CUDA memcpy DtoH]
                    7.28%  113.66ms         2  56.830ms  56.760ms  56.899ms  [CUDA memcpy HtoD]
      API calls:   69.79%  1.22736s         2  613.68ms  82.132us  1.22728s  cudaDeviceSynchronize
                   19.07%  335.32ms         3  111.77ms  56.938ms  221.25ms  cudaMemcpy
                   10.62%  186.72ms         4  46.680ms     544ns  186.71ms  cudaEventCreate
                    0.28%  4.9182ms       228  2

In [7]:
!nvprof ./matmul_tiling 16384

Running GPU naive matmul for N = 16384
==265== NVPROF is profiling process 265, command: ./matmul_tiling 16384
Kernel Execution time: 9137.666 ms
Total execution time on GPU: 10475.845 ms
Works Correctly
==265== Profiling application: ./matmul_tiling 16384
==265== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   87.25%  9.13730s         1  9.13730s  9.13730s  9.13730s  matmul_naive(float*, float*, float*, int)
                    8.39%  879.14ms         1  879.14ms  879.14ms  879.14ms  [CUDA memcpy DtoH]
                    4.36%  456.11ms         2  228.05ms  225.70ms  230.41ms  [CUDA memcpy HtoD]
      API calls:   85.60%  9.13739s         2  4.56870s  79.305us  9.13731s  cudaDeviceSynchronize
                   12.53%  1.33739s         3  445.80ms  225.93ms  880.87ms  cudaMemcpy
                    1.75%  187.23ms         4  46.807ms     605ns  187.22ms  cudaEventCreate
                    0.06%  5.8951ms         

In [8]:
!nvprof ./matmul_tiling 32768

Running GPU naive matmul for N = 32768
==289== NVPROF is profiling process 289, command: ./matmul_tiling 32768
Kernel Execution time: 79682.867 ms
Total execution time on GPU: 85055.391 ms
Works Correctly
==289== Profiling application: ./matmul_tiling 32768
==289== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   93.69%  79.6820s         1  79.6820s  79.6820s  79.6820s  matmul_naive(float*, float*, float*, int)
                    4.19%  3.56438s         1  3.56438s  3.56438s  3.56438s  [CUDA memcpy DtoH]
                    2.12%  1.80482s         2  902.41ms  896.86ms  907.96ms  [CUDA memcpy HtoD]
      API calls:   93.46%  79.6821s         2  39.8411s  76.464us  79.6820s  cudaDeviceSynchronize
                    6.30%  5.37132s         3  1.79044s  897.03ms  3.56610s  cudaMemcpy
                    0.22%  189.90ms         4  47.475ms     593ns  189.88ms  cudaEventCreate
                    0.01%  11.291ms        

In [17]:
%%writefile matmulk.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

#define TILE 32
__global__ void matmul(float *fA,float *fB,float*fC,int N) {
    int globalcol=threadIdx.x+blockDim.x*blockIdx.x;
    int globalrow=threadIdx.y+blockDim.y*blockIdx.y;
    __shared__ float As[TILE*TILE];
    __shared__ float Bs[TILE*TILE];
    int localcol=threadIdx.x;
    int localrow=threadIdx.y;
    int num_tiles=(N+TILE-1)/TILE;
    float sumt=0.0f;

    for(int y=0;y<num_tiles;y++){
        int Arow=globalrow;
        int Acol=y*TILE+localcol;
        int Brow=y*TILE+localrow;
        int Bcol=globalcol;
        if(Arow<N && Acol<N){
             As[localrow*TILE+localcol]=fA[Arow*N+Acol];
        }
        else{
            As[localrow*TILE+localcol]=0.0f;
        }
        if(Brow<N && Bcol<N){
            Bs[localrow*TILE+localcol]=fB[Brow*N+Bcol];
        }
        else{
            Bs[localrow*TILE+localcol]=0.0f;
        }
        __syncthreads();

        for(int i=0;i<TILE;i++){
            sumt+=As[localrow*TILE+i]*Bs[i*TILE+localcol];
        }
        __syncthreads();
    }
    if(globalrow<N && globalcol<N){
        fC[globalrow*N+globalcol]=sumt;
    }
}

using namespace std;

int main() {
    int N=1024;
    dim3 threadPerBlock(32,32);
    int t=(N+31)/32;
    dim3 blocks(t,t);
    size_t size = N * N * sizeof(float);
    float *A=(float*)malloc(size);
    float *B=(float*)malloc(size);
    float *C=(float*)malloc(size);
    for(int i=0;i<N*N;i++){
        A[i]=1.0f;
        B[i]=1.0f;
    }
    
    cudaEvent_t estart,estop,fstart,fstop;
    cudaEventCreate(&estart);
    cudaEventCreate(&estop);
    cudaEventCreate(&fstart);
    cudaEventCreate(&fstop);
    cudaEventRecord(estart);

    float *fA,*fB,*fC;
    cudaMalloc(&fA,size);
    cudaMalloc(&fB,size);
    cudaMalloc(&fC,size);
    cudaMemcpy(fA,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(fB,B,size,cudaMemcpyHostToDevice);

    cudaDeviceSynchronize();

    cudaEventRecord(fstart);
    matmul<<<blocks,threadPerBlock>>>(fA,fB,fC,N);
    cudaEventRecord(fstop);

    cudaMemcpy(C,fC,size,cudaMemcpyDeviceToHost);

    cudaEventRecord(estop);

    cudaDeviceSynchronize();

    float kernelmilliseconds = 0;
    cudaEventElapsedTime(&kernelmilliseconds, fstart, fstop);
    printf("Kernel Execution time: %.3f ms\n", kernelmilliseconds);

    float overallmilliseconds = 0;
    cudaEventElapsedTime(&overallmilliseconds, estart, estop);
    printf("Total execution time on GPU: %.3f ms\n", overallmilliseconds);

    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            if(fabs(C[i*N+j]-N) > 1e-5){
                fprintf(stderr, "Result verification failed at element %d!\n", i);
                cudaFree(fA);
                cudaFree(fB);
                cudaFree(fC);
                free(A);
                free(B);
                free(C);
                exit(EXIT_FAILURE);
            }
        }
    }
    cout<<"Works Correctly"<<endl;
    cudaFree(fA);
    cudaFree(fB);
    cudaFree(fC);
    free(A);
    free(B);
    free(C);

    return 0;
}


Overwriting matmulk.cu


In [18]:
!nvcc matmulk.cu -o matmulk

In [19]:
!nvprof ./matmulk

==335== NVPROF is profiling process 335, command: ./matmulk
Kernel Execution time: 26.408 ms
Total execution time on GPU: 32.678 ms
Works Correctly
==335== Profiling application: ./matmulk
==335== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   54.65%  4.5408ms         1  4.5408ms  4.5408ms  4.5408ms  matmul(float*, float*, float*, int)
                   25.52%  2.1207ms         1  2.1207ms  2.1207ms  2.1207ms  [CUDA memcpy DtoH]
                   19.83%  1.6476ms         2  823.78us  821.05us  826.52us  [CUDA memcpy HtoD]
      API calls:   82.58%  182.95ms         4  45.738ms     483ns  182.94ms  cudaEventCreate
                    9.87%  21.865ms         1  21.865ms  21.865ms  21.865ms  cudaLaunchKernel
                    4.67%  10.345ms         3  3.4482ms  985.85us  8.2984ms  cudaMemcpy
                    2.32%  5.1314ms       228  22.506us     105ns  1.5056ms  cuDeviceGetAttribute
                    0.33%

In [25]:
%%writefile matmul_tiling_register.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cstdlib>
#include <iostream>

#define TILE 32
#define COL_PER_THREAD 4

__global__ void matmul_naive(float* A,float* B,float* C,int N){
    __shared__ float As[TILE*TILE];
    __shared__ float Bs[TILE*TILE*COL_PER_THREAD];
    int baseGlobalCol = blockIdx.x * TILE * COL_PER_THREAD;
    int globalcol = baseGlobalCol + threadIdx.x;
    int globalrow=blockIdx.y*blockDim.y+threadIdx.y;

    int localrow=threadIdx.y;
    int localcol=threadIdx.x;
    int NUM_TILES=(N+TILE-1)/TILE;
    
    float sum[COL_PER_THREAD]={0.0f,0.0f,0.0f,0.0f};
    
    for(int i=0;i<NUM_TILES;i++){
        int Arow=globalrow;
        int Acol=i*TILE+localcol;
        int Brow=i*TILE+localrow;
        if(Arow<N && Acol<N){
            As[localrow*TILE+localcol]=A[Arow*N+Acol]; 
        }
        else{
            As[localrow*TILE+localcol]=0.0f;
        }
        for(int y=0;y<COL_PER_THREAD;y++){
            int Bcol=globalcol+TILE*y;
            if(Brow<N && Bcol<N){
                Bs[y*TILE*TILE+localrow*TILE+localcol]=B[Brow*N+Bcol];
            }
            else{
                Bs[y*TILE*TILE+localrow*TILE+localcol]=0.0f;
            }
        }
        
        __syncthreads();
        #pragma unroll
        for(int k=0;k<TILE;k++){
            float a=As[localrow*TILE+k];
            #pragma unroll
            for(int y=0;y<COL_PER_THREAD;y++){
                sum[y]+=a*Bs[y*TILE*TILE+k*TILE+localcol];
            }
        }        
        __syncthreads();
    }
    for(int y=0;y<COL_PER_THREAD;y++){
        int gcol = globalcol + y*TILE;
        if(globalrow < N && gcol < N)
            C[globalrow*N + gcol] = sum[y];
    }
}

using namespace std; 

int main(int argc,char* argv[]){
    if (argc < 2) {
        printf("Usage: ./matmul_naive N\n");
        return 1;
    }

    int N = atoi(argv[1]);
    size_t size = N * N * sizeof(float);

    printf("Running GPU naive matmul for N = %d\n", N);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N*N; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 1.0f;
    }
    cudaEvent_t start, stop, start_overall, stop_overall;
    cudaEventCreate(&start_overall);
    cudaEventCreate(&stop_overall);
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    cudaEventRecord(start_overall);
    
    float *d_A,*d_B,*d_C;
    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);
    
    cudaMemcpy(d_A,h_A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,size,cudaMemcpyHostToDevice);

    dim3 ThreadsPerBlock(32,32);
    dim3 BlocksPerGrid(
        (N + TILE*COL_PER_THREAD - 1)/(TILE*COL_PER_THREAD),
        (N + TILE - 1)/TILE
    );

    cudaDeviceSynchronize();

    cudaEventRecord(start);
    matmul_naive<<<BlocksPerGrid,ThreadsPerBlock>>>(d_A,d_B,d_C,N);
    cudaEventRecord(stop);

    cudaDeviceSynchronize();
    cudaMemcpy(h_C,d_C,size,cudaMemcpyDeviceToHost);
    
    cudaEventRecord(stop_overall);

    cudaEventSynchronize(stop);
    float kernelmilliseconds = 0;
    cudaEventElapsedTime(&kernelmilliseconds, start, stop);
    printf("Kernel Execution time: %.3f ms\n", kernelmilliseconds);

    cudaEventSynchronize(stop_overall);
    float overallmilliseconds = 0;
    cudaEventElapsedTime(&overallmilliseconds, start_overall, stop_overall);
    printf("Total execution time on GPU: %.3f ms\n", overallmilliseconds);
    
    cudaEventDestroy(start_overall);
    cudaEventDestroy(stop_overall);
    
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            if(h_C[i*N+j]-N > 1e-5){
                fprintf(stderr, "Result verification failed at element %d!\n", i);
                cudaFree(d_A);
                cudaFree(d_B);
                cudaFree(d_C);
                free(h_A);
                free(h_B);
                free(h_C);
                exit(EXIT_FAILURE);
            }
        }
    }
    cout<<"Works Correctly"<<endl;
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);
    return 0;
}

Overwriting matmul_tiling_register.cu


In [26]:
!nvcc matmul_tiling_register.cu -o matmul_tiling_register

In [27]:
!nvprof ./matmul_tiling_register 1024

Running GPU naive matmul for N = 1024
==874== NVPROF is profiling process 874, command: ./matmul_tiling_register 1024
Kernel Execution time: 39.700 ms
Total execution time on GPU: 45.762 ms
Works Correctly
==874== Profiling application: ./matmul_tiling_register 1024
==874== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   53.33%  3.9903ms         1  3.9903ms  3.9903ms  3.9903ms  matmul_naive(float*, float*, float*, int)
                   25.22%  1.8872ms         1  1.8872ms  1.8872ms  1.8872ms  [CUDA memcpy DtoH]
                   21.45%  1.6055ms         2  802.73us  802.07us  803.38us  [CUDA memcpy HtoD]
      API calls:   79.21%  196.45ms         4  49.113ms     549ns  196.45ms  cudaEventCreate
                   14.40%  35.709ms         1  35.709ms  35.709ms  35.709ms  cudaLaunchKernel
                    2.21%  5.4904ms         3  1.8301ms  957.82us  3.5054ms  cudaMemcpy
                    2.05%  5.0891ms    

In [28]:
!nvprof ./matmul_tiling_register 32768

Running GPU naive matmul for N = 32768
==898== NVPROF is profiling process 898, command: ./matmul_tiling_register 32768
Kernel Execution time: 50035.145 ms
Total execution time on GPU: 55611.715 ms
Works Correctly
==898== Profiling application: ./matmul_tiling_register 32768
==898== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   89.98%  50.0342s         1  50.0342s  50.0342s  50.0342s  matmul_naive(float*, float*, float*, int)
                    6.62%  3.68302s         1  3.68302s  3.68302s  3.68302s  [CUDA memcpy DtoH]
                    3.40%  1.89002s         2  945.01ms  941.96ms  948.06ms  [CUDA memcpy HtoD]
      API calls:   89.64%  50.0343s         2  25.0171s  75.914us  50.0342s  cudaDeviceSynchronize
                    9.99%  5.57522s         3  1.85841s  942.19ms  3.68477s  cudaMemcpy
                    0.34%  190.25ms         4  47.563ms     683ns  190.24ms  cudaEventCreate
                    0.02%

In [29]:
%%writefile matmul_tiling_register2.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cstdlib>
#include <iostream>

#define TILE 32
#define ROW_PER_THREAD 4

__global__ void matmul_naive(float* A,float* B,float* C,int N){
    __shared__ float As[TILE*TILE*ROW_PER_THREAD];
    __shared__ float Bs[TILE*TILE];
    int baseGlobalrow = blockIdx.y * TILE * ROW_PER_THREAD;
    int globalrow = baseGlobalrow + threadIdx.y;
    int globalcol=blockIdx.x*blockDim.x+threadIdx.x;

    int localrow=threadIdx.y;
    int localcol=threadIdx.x;
    int NUM_TILES=(N+TILE-1)/TILE;
    
    float sum[ROW_PER_THREAD]={0.0f,0.0f,0.0f,0.0f};
    
    for(int i=0;i<NUM_TILES;i++){
        int Acol=i*TILE+localcol;
        int Brow=i*TILE+localrow;
        int Bcol=globalcol;
        if(Brow<N && Bcol<N){
            Bs[localrow*TILE+localcol]=B[Brow*N+Bcol];
        }
        else{
            Bs[localrow*TILE+localcol]=0.0f;
        }
        
        for(int y=0;y<ROW_PER_THREAD;y++){
            int Arow=globalrow+TILE*y;
            if(Arow<N && Acol<N){
                As[y*TILE*TILE+localrow*TILE+localcol]=A[Arow*N+Acol]; 
            }
            else{
                As[y*TILE*TILE+localrow*TILE+localcol]=0.0f;
            }
        }
        
        __syncthreads();
        #pragma unroll
        for(int k=0;k<TILE;k++){
            float b=Bs[k*TILE+localcol];
            #pragma unroll
            for(int y=0;y<ROW_PER_THREAD;y++){
                sum[y]+=As[y*TILE*TILE+localrow*TILE+k]*b;
            }
        }        
        __syncthreads();
    }
    for(int y=0;y<ROW_PER_THREAD;y++){
        int grow = globalrow + y*TILE;
        if(grow < N && globalcol < N)
            C[grow*N + globalcol] = sum[y];
    }
}

using namespace std; 

int main(int argc,char* argv[]){
    if (argc < 2) {
        printf("Usage: ./matmul_naive N\n");
        return 1;
    }

    int N = atoi(argv[1]);
    size_t size = N * N * sizeof(float);

    printf("Running GPU naive matmul for N = %d\n", N);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N*N; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 1.0f;
    }
    cudaEvent_t start, stop, start_overall, stop_overall;
    cudaEventCreate(&start_overall);
    cudaEventCreate(&stop_overall);
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    cudaEventRecord(start_overall);
    
    float *d_A,*d_B,*d_C;
    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);
    
    cudaMemcpy(d_A,h_A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,size,cudaMemcpyHostToDevice);

    dim3 ThreadsPerBlock(32,32);
    dim3 BlocksPerGrid(
        (N + TILE - 1)/TILE,(N + TILE*ROW_PER_THREAD - 1)/(TILE*ROW_PER_THREAD)
    );

    cudaDeviceSynchronize();

    cudaEventRecord(start);
    matmul_naive<<<BlocksPerGrid,ThreadsPerBlock>>>(d_A,d_B,d_C,N);
    cudaEventRecord(stop);

    cudaDeviceSynchronize();
    cudaMemcpy(h_C,d_C,size,cudaMemcpyDeviceToHost);
    
    cudaEventRecord(stop_overall);

    cudaEventSynchronize(stop);
    float kernelmilliseconds = 0;
    cudaEventElapsedTime(&kernelmilliseconds, start, stop);
    printf("Kernel Execution time: %.3f ms\n", kernelmilliseconds);

    cudaEventSynchronize(stop_overall);
    float overallmilliseconds = 0;
    cudaEventElapsedTime(&overallmilliseconds, start_overall, stop_overall);
    printf("Total execution time on GPU: %.3f ms\n", overallmilliseconds);
    
    cudaEventDestroy(start_overall);
    cudaEventDestroy(stop_overall);
    
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            if(h_C[i*N+j]-N > 1e-5){
                fprintf(stderr, "Result verification failed at element %d!\n", i);
                cudaFree(d_A);
                cudaFree(d_B);
                cudaFree(d_C);
                free(h_A);
                free(h_B);
                free(h_C);
                exit(EXIT_FAILURE);
            }
        }
    }
    cout<<"Works Correctly"<<endl;
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);
    return 0;
}

Writing matmul_tiling_register2.cu


In [31]:
!nvcc matmul_tiling_register2.cu -o matmul_reg2

In [32]:
!nvprof ./matmul_reg2 1024

Running GPU naive matmul for N = 1024
==1299== NVPROF is profiling process 1299, command: ./matmul_reg2 1024
Kernel Execution time: 35.473 ms
Total execution time on GPU: 41.675 ms
Works Correctly
==1299== Profiling application: ./matmul_reg2 1024
==1299== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   35.54%  1.9284ms         1  1.9284ms  1.9284ms  1.9284ms  matmul_naive(float*, float*, float*, int)
                   35.05%  1.9018ms         1  1.9018ms  1.9018ms  1.9018ms  [CUDA memcpy DtoH]
                   29.41%  1.5955ms         2  797.75us  774.62us  820.89us  [CUDA memcpy HtoD]
      API calls:   79.76%  186.94ms         4  46.734ms     454ns  186.93ms  cudaEventCreate
                   14.33%  33.589ms         1  33.589ms  33.589ms  33.589ms  cudaLaunchKernel
                    2.46%  5.7591ms         3  1.9197ms  931.86us  3.7890ms  cudaMemcpy
                    2.15%  5.0456ms       228  22.129us  

In [33]:
!nvprof ./matmul_reg2 4096

Running GPU naive matmul for N = 4096
==1310== NVPROF is profiling process 1310, command: ./matmul_reg2 4096
Kernel Execution time: 139.395 ms
Total execution time on GPU: 224.784 ms
Works Correctly
==1310== Profiling application: ./matmul_reg2 4096
==1310== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   62.68%  139.06ms         1  139.06ms  139.06ms  139.06ms  matmul_naive(float*, float*, float*, int)
                   24.36%  54.053ms         1  54.053ms  54.053ms  54.053ms  [CUDA memcpy DtoH]
                   12.95%  28.741ms         2  14.370ms  14.167ms  14.574ms  [CUDA memcpy HtoD]
      API calls:   44.20%  184.01ms         4  46.002ms     475ns  184.01ms  cudaEventCreate
                   33.43%  139.17ms         2  69.583ms  89.900us  139.08ms  cudaDeviceSynchronize
                   20.39%  84.864ms         3  28.288ms  14.381ms  55.748ms  cudaMemcpy
                    1.18%  4.9131ms       228  21.

In [34]:
!nvprof ./matmul_reg2 32768

Running GPU naive matmul for N = 32768
==1321== NVPROF is profiling process 1321, command: ./matmul_reg2 32768
Kernel Execution time: 35890.867 ms
Total execution time on GPU: 41413.684 ms
Works Correctly
==1321== Profiling application: ./matmul_reg2 32768
==1321== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   86.67%  35.8901s         1  35.8901s  35.8901s  35.8901s  matmul_naive(float*, float*, float*, int)
                    8.81%  3.65009s         1  3.65009s  3.65009s  3.65009s  [CUDA memcpy DtoH]
                    4.51%  1.86933s         2  934.66ms  930.95ms  938.38ms  [CUDA memcpy HtoD]
      API calls:   86.23%  35.8902s         2  17.9451s  74.273us  35.8901s  cudaDeviceSynchronize
                   13.27%  5.52159s         3  1.84053s  931.12ms  3.65177s  cudaMemcpy
                    0.46%  192.30ms         4  48.075ms     482ns  192.30ms  cudaEventCreate
                    0.03%  11.257ms        